In [1]:
import re
from collections import defaultdict
import json 
from typing import List, Tuple
from copy import deepcopy

import evaluate
from torch.utils.data import DataLoader,Dataset
import transformers
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import torch
import numpy as np
import pandas as pd
import wandb

from new_module.new_decode_utils import get_beam_hypotheses_v0, CustomDataset, repeat_interleave_unravel, analyze_span_lengths_and_count
import new_module.losses as lossbuilder

/data/hyeryung/.conda/envs/loc-edit/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


### BertScore

In [6]:
### lossfn 객체 생성
toxicity_em_path = '/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint'

build_loss_dict = \
{'AR_top_k': 0,
'AR_top_p': 0.96,
'loss_type': 'xentropy',
'coeff_steps': 200,
'coeff_pattern': 'constant',
'AR_temperature': 1,
'length_normalize': False,
'max_output_length': 20}

class dummyArgs:
        def __init__(self, **kwargs):
            for k, v in kwargs.items():
                setattr(self, k, v)

build_loss_args = dummyArgs(**build_loss_dict)
build_loss_args.task = "toxicity"

model = AutoModelForSequenceClassification.from_pretrained(toxicity_em_path)
tokenizer = AutoTokenizer.from_pretrained(toxicity_em_path)

lossfn = lossbuilder.build_loss(
            'bertscore',
            model = model,
            tokenizer = tokenizer,
            args = build_loss_args
        )
mlm_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
lossfn.tokenizer.add_special_tokens({"mask_token": mlm_tokenizer.mask_token})


Starting new HTTPS connection (1): s3.amazonaws.com:443
https://s3.amazonaws.com:443 "HEAD /datasets.huggingface.co/datasets/metrics/evaluate-metric/bertscore/evaluate-metric/bertscore.py HTTP/11" 404 0
Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /spaces/evaluate-metric/bertscore/resolve/v0.4.3/bertscore.py HTTP/11" 404 0
Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /spaces/evaluate-metric/bertscore/resolve/main/bertscore.py HTTP/11" 307 0
https://huggingface.co:443 "HEAD /api/resolve-cache/spaces/evaluate-metric/bertscore/61f4e3e5ec826799d2496bb42ca9626efcb560d5/bertscore.py?%2Fspaces%2Fevaluate-metric%2Fbertscore%2Fresolve%2Fmain%2Fbertscore.py=&etag=%22071e76ff3be0ad52302ab4d5fc99beb4ed125161%22 HTTP/11" 200 0
Attempting to acquire lock 139990567578256 on /data/hyeryung/hf_cache/modules/evaluate_modules/metrics/evaluate-metric--bertscore.lock
Lock 139990567578256 acquired on /data/hyeryung/hf_cache/

1

In [ ]:
### original 문장과 test 문장 정의
original_sents = ["hello world. Hello world! herro WORLD!!", "Cat hat bat sat pat mat."]
edited_sents = ["hello world! Hello world. herro WORld.", "CAT sat on a mat."]

In [19]:
### lossfn에 통과시켜 값 뽑기 (A)
value_from_lossfn = lossfn.compute_gold_loss("START", original_sents, references=edited_sents); value_from_lossfn

Attempting to acquire lock 139992184386800 on /data/hyeryung/hf_cache/metrics/bert_score/default/default_experiment-1-0.arrow.lock
Lock 139992184386800 acquired on /data/hyeryung/hf_cache/metrics/bert_score/default/default_experiment-1-0.arrow.lock
open file: /data/hyeryung/hf_cache/metrics/bert_score/default/default_experiment-1-0.arrow
Attempting to release lock 139992184386800 on /data/hyeryung/hf_cache/metrics/bert_score/default/default_experiment-1-0.arrow.lock
Lock 139992184386800 released on /data/hyeryung/hf_cache/metrics/bert_score/default/default_experiment-1-0.arrow.lock


tensor(-0.436)

In [26]:
### 외부에서 같은 로직을 구현 (B)
bertscore = evaluate.load('bertscore')
print(bertscore.compute(predictions=original_sents, 
          references = edited_sents, lang="en", rescale_with_baseline=True)['recall'])
value_from_evaluate = -1 * np.mean(bertscore.compute(predictions=original_sents, 
          references = edited_sents, lang="en", rescale_with_baseline=True)['recall'])
print(value_from_evaluate)

Starting new HTTPS connection (1): s3.amazonaws.com:443
https://s3.amazonaws.com:443 "HEAD /datasets.huggingface.co/datasets/metrics/evaluate-metric/bertscore/evaluate-metric/bertscore.py HTTP/11" 404 0
Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /spaces/evaluate-metric/bertscore/resolve/v0.4.3/bertscore.py HTTP/11" 404 0
Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /spaces/evaluate-metric/bertscore/resolve/main/bertscore.py HTTP/11" 307 0
https://huggingface.co:443 "HEAD /api/resolve-cache/spaces/evaluate-metric/bertscore/61f4e3e5ec826799d2496bb42ca9626efcb560d5/bertscore.py?%2Fspaces%2Fevaluate-metric%2Fbertscore%2Fresolve%2Fmain%2Fbertscore.py=&etag=%22071e76ff3be0ad52302ab4d5fc99beb4ed125161%22 HTTP/11" 200 0
Attempting to acquire lock 139991910458080 on /data/hyeryung/hf_cache/modules/evaluate_modules/metrics/evaluate-metric--bertscore.lock
Lock 139991910458080 acquired on /data/hyeryung/hf_cache/

[0.4658549726009369, 0.4069931209087372]
-0.43642404675483704


In [ ]:
assert value_from_lossfn == value_from_evaluate

### Edit Distance

In [2]:
### lossfn 객체 생성
toxicity_em_path = '/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint'

build_loss_dict = \
{'AR_top_k': 0,
'AR_top_p': 0.96,
'loss_type': 'xentropy',
'coeff_steps': 200,
'coeff_pattern': 'constant',
'AR_temperature': 1,
'length_normalize': False,
'max_output_length': 20}

class dummyArgs:
        def __init__(self, **kwargs):
            for k, v in kwargs.items():
                setattr(self, k, v)

build_loss_args = dummyArgs(**build_loss_dict)
build_loss_args.task = "toxicity"

model = AutoModelForSequenceClassification.from_pretrained(toxicity_em_path)
tokenizer = AutoTokenizer.from_pretrained(toxicity_em_path)

lossfn = lossbuilder.build_loss(
            'edit_distance',
            model = model,
            tokenizer = tokenizer,
            args = build_loss_args
        )
mlm_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
lossfn.tokenizer.add_special_tokens({"mask_token": mlm_tokenizer.mask_token})


Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /roberta-base/resolve/main/tokenizer_config.json HTTP/11" 200 0


1

In [3]:
### original 문장과 test 문장 정의
original_sents = ["hello world. Hello world! herro WORLD!!", "Cat hat bat sat pat mat."]
edited_sents = ["hello world! Hello world. herro WORld.", "CAT sat on a mat."]


In [4]:
### lossfn에 통과시켜 값 뽑기 (A)
value_from_lossfn = lossfn.compute_gold_loss(prompt="START", predictions=edited_sents, references=original_sents); value_from_lossfn


tensor(0.584)

In [9]:
### 외부에서 같은 로직을 구현 (B)
value_from_evaluate = None 
normalize = True
predictions = edited_sents
references = original_sents

def _levenshtein(a_seq, b_seq):
    """
    Compute Levenshtein distance between two sequences (list of tokens or chars).
    Uses the classic DP with two rolling rows. O(len(a)*len(b)) time, O(min) space.
    """
    len_a, len_b = len(a_seq), len(b_seq)
    # Ensure the shorter sequence is on the horizontal axis to reduce memory
    if len_a < len_b:
        a_seq, b_seq = b_seq, a_seq
        len_a, len_b = len_b, len_a

    # previous[j] = distance between a_seq[:i-1] and b_seq[:j]
    previous = list(range(len_b + 1))
    for i in range(1, len_a + 1):
        current = [i]
        ai = a_seq[i - 1]
        for j in range(1, len_b + 1):
            cost_sub = 0 if ai == b_seq[j - 1] else 1
            insert_cost = current[j - 1] + 1
            delete_cost = previous[j] + 1
            replace_cost = previous[j - 1] + cost_sub
            current.append(min(insert_cost, delete_cost, replace_cost))
        previous = current
    return previous[-1]

n = len(predictions); print(n)
total = 0.0
all_list = []
for pred, ref in zip(predictions, references):
    # Basic sanitation
    pred = "" if pred is None else str(pred)
    ref = "" if ref is None else str(ref)
    
    a_seq = lossfn.tokenizer.tokenize(pred)
    b_seq = lossfn.tokenizer.tokenize(ref)

    dist = _levenshtein(a_seq, b_seq)

    if normalize:
        denom = max(len(a_seq), len(b_seq))
        dist = (dist / denom) if denom > 0 else 0.0

    total += dist
    all_list.append(dist)

avg_dist = total / n


2


In [10]:
value_from_evaluate = avg_dist

In [11]:
avg_dist

0.5844155844155844

In [19]:
value_from_lossfn.cpu()

tensor(0.584)

In [14]:
value_from_evaluate

0.5844155844155844

In [16]:
assert value_from_lossfn == value_from_evaluate